# Build Train/Test Processed Data

Creates leakage-aware processed artifacts for model evaluation.

- Train: 2021-2024
- Test: 2025 holdout
- Fit preprocessing and PCA on train only
- Transform 2025 using train-fitted parameters


## Imports and Configuration


In [29]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


YEARS = range(2021, 2026)
TRAIN_END_YEAR = 2024
TEST_YEAR = 2025

FEATURES = [
    "avail_rate",
    "blk",
    "fg",
    "fg_per_g",
    "fga",
    "ft",
    "fta",
    "g",
    "mp",
    "pca1",
    "pts",
    "pts_rookie",
    "pts_vet",
    "pts_hardship",
    "start_rate",
    "team_min",
    "tov",
    "ws_rookie",
    "ws_vet",
    "ws_hardship",
    "group",
]

PCA_COLS = ["mp", "pts", "fg", "fga", "ft", "trb", "ast", "ws", "pts40", "ast40"]

SEASON_GAMES = {
    2021: 32,
    2022: 36,
    2023: 40,
    2024: 40,
    2025: 44,
}

TEAMMAP = {
    "Atlanta Dream": "ATL",
    "Chicago Sky": "CHI",
    "Connecticut Sun": "CON",
    "Dallas Wings": "DAL",
    "Golden State Valkyries": "GSV",
    "Indiana Fever": "IND",
    "Las Vegas Aces": "LVA",
    "Los Angeles Sparks": "LAS",
    "Minnesota Lynx": "MIN",
    "New York Liberty": "NYL",
    "Phoenix Mercury": "PHO",
    "Seattle Storm": "SEA",
    "Washington Mystics": "WAS",
}

NAME_FIX = {
    "AD Durr": "Asia (AD) Durr",
    "Anastasiia Kosu": "Anastasiia Olairi Kosu",
    "Azurá Stevens": "Azura Stevens",
    "Azura Stevens": "Azura Stevens",
    "Brittany Boyd": "Brittany Boyd-Jones",
    "Dorka JuhÃ¡sz": "Dorka Juhasz",
    "Dorka Juhász": "Dorka Juhasz",
    "Ivana DojkiÄ": "Ivana Dojkic",
    "Ivana DojkiÄ": "Ivana Dojkic",
    "Ivana Dojkić": "Ivana Dojkic",
    "Janelle SalaÃ¼n": "Janelle Salaun",
    "Janelle Salaün": "Janelle Salaun",
    "LeÃ¯la Lacan": "Leila Lacan",
    "Leïla Lacan": "Leila Lacan",
    "Lou Lopez SÃ©nÃ©chal": "Lou Lopez Senechal",
    "Lou Lopez Sénéchal": "Lou Lopez Senechal",
    "Luisa GeiselsÃ¶der": "Luisa Geiselsoder",
    "Luisa Geiselsöder": "Luisa Geiselsoder",
    "Mamignan TourÃ©": "Mamignan Toure",
    "Mamignan Touré": "Mamignan Toure",
    "MariÃ¨me Badiane": "Marieme Badiane",
    "Marième Badiane": "Marieme Badiane",
    "Nika MÃ¼hl": "Nika Muhl",
    "Nika Mühl": "Nika Muhl",
    "Olivia Ãpoupa": "Olivia Epoupa",
    "Olivia Ãpoupa": "Olivia Epoupa",
    "Olivia Époupa": "Olivia Epoupa",
    "Sika KonÃ©": "Sika Kone",
    "Sika Koné": "Sika Kone",
    "Te-Hina PaoPao": "Te-Hina Paopao",
    "Temi Fágbénlé": "Temi Fagbenle",
}


## Cleaning and Merge Helpers


In [30]:
def clean_col(name):
    text = str(name).strip().lower()
    text = re.sub(r"\b202\d\b", "", text)
    text = text.strip()
    if "salary" in text:
        text = "salary"
    if "signing" in text:
        text = "signing"
    text = text.replace("%", "pct")
    text = re.sub(r"[^0-9a-z]+", "_", text)
    return text.strip("_")


def clean_df(df):
    df = df.copy()
    df.columns = [clean_col(col) for col in df.columns]
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .replace({"": np.nan, "—": np.nan, "nan": np.nan})
        )
    return df


def team_clean(value):
    if pd.isna(value):
        return np.nan
    return str(value).replace("*", "").strip()


def to_numeric(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def div(a, b):
    a = pd.to_numeric(pd.Series(a), errors="coerce")
    b = pd.to_numeric(pd.Series(b), errors="coerce")
    return np.where(b != 0, a / b, np.nan)


def get_group(value):
    if pd.isna(value):
        return "unknown"
    value = str(value).strip().lower()
    if "rookie" in value:
        return "rookie"
    if "hardship" in value or "susp" in value or value == "7d":
        return "hardship"
    if value in ["ufa", "rfa", "core"]:
        return "veteran"
    if value in ["udfa", "reserved", "ros"]:
        return "controlled"
    return "other"


def load_merge(raw_dir):
    season_dfs = []

    for year in YEARS:
        adv = clean_df(pd.read_csv(raw_dir / f"{year}_advanced.csv"))
        per = clean_df(pd.read_csv(raw_dir / f"{year}_per_game.csv"))
        tot = clean_df(pd.read_csv(raw_dir / f"{year}_totals.csv"))
        sal = clean_df(pd.read_csv(raw_dir / f"salary_{year}.csv"))
        teamadv = clean_df(pd.read_csv(raw_dir / f"{year}_advanced-team.csv"))
        stand = clean_df(pd.read_csv(raw_dir / f"{year}_wnba_standings.csv"))

        sal["salary"] = pd.to_numeric(sal["salary"], errors="coerce")

        teamadv["team_name"] = teamadv["team"].map(team_clean)
        teamadv["team"] = teamadv["team_name"].map(TEAMMAP)
        stand["team_name"] = stand["team_name"].map(team_clean)
        stand["team"] = stand["team_name"].map(TEAMMAP)

        teamdf = teamadv.merge(
            stand,
            on=["team", "team_name"],
            how="left",
            suffixes=("_adv", "_stand"),
        )

        playerdf = per.merge(
            adv,
            on=["player", "team", "pos", "g", "mp"],
            how="outer",
            suffixes=("_per", "_adv"),
        )
        playerdf = playerdf.merge(
            tot,
            on=["player", "team", "pos", "g", "mp", "gs"],
            how="outer",
            suffixes=("", "_tot"),
        )

        sal["player"] = sal["player"].replace(NAME_FIX)
        playerdf["player"] = playerdf["player"].replace(NAME_FIX)

        seasondf = sal.merge(playerdf, on="player", how="inner", suffixes=("_sal", ""))
        seasondf = seasondf.merge(teamdf, on="team", how="left", suffixes=("", "_team"))
        seasondf["year"] = year
        season_dfs.append(seasondf.copy())

    final_df = pd.concat(season_dfs, ignore_index=True)
    return final_df.drop(columns=["dummy_x", "dummy_y"], errors="ignore")


## Feature Engineering


In [31]:
def add_features(df):
    df = df.copy()

    numeric_cols = set(PCA_COLS + [c for c in FEATURES if c != "group"])
    numeric_cols.update(
        [
            "salary",
            "g",
            "gs",
            "mp",
            "mp_per_g",
            "pts_per_g",
            "fga_per_g",
            "fta_per_g",
            "fg3a_per_g",
            "ast_per_g",
            "trb_per_g",
            "stl_per_g",
            "blk_per_g",
            "tov_per_g",
            "ws",
            "ts_pct_x",
            "ts_pct_y",
            "efg_pct_x",
            "efg_pct_y",
            "win_loss_pct",
            "net_rtg",
            "srs",
        ]
    )
    df = to_numeric(df, numeric_cols)

    df["group"] = df["signing"].apply(get_group)
    df["rookie_flag"] = (df["group"] == "rookie").astype(int)
    df["hardship_flag"] = (df["group"] == "hardship").astype(int)
    df["vet_flag"] = (df["group"] == "veteran").astype(int)
    df["unknown_flag"] = (df["group"] == "unknown").astype(int)

    season_games = df["year"].map(SEASON_GAMES)
    df["avail_rate"] = div(df["g"], season_games)
    df["start_rate"] = np.where(df["g"] > 0, df["gs"] / df["g"], 0)
    df["team_min"] = div(df["mp"], season_games)
    df["starter"] = (df["start_rate"] >= 0.5).astype(int)
    df["rotation"] = (df["mp_per_g"] >= 15).astype(int)

    per40 = div(40, df["mp_per_g"])
    df["pts40"] = df["pts_per_g"] * per40
    df["ast40"] = df["ast_per_g"] * per40
    df["reb40"] = df["trb_per_g"] * per40
    df["stocks"] = df["stl_per_g"] + df["blk_per_g"]
    df["stocks40"] = df["stocks"] * per40
    df["pps"] = div(df["pts_per_g"], df["fga_per_g"] + 0.44 * df["fta_per_g"])
    df["ft_rate"] = div(df["fta_per_g"], df["fga_per_g"])
    df["three_rate"] = div(df["fg3a_per_g"], df["fga_per_g"])
    df["ast_tov"] = div(df["ast_per_g"], df["tov_per_g"])
    df["ws_game"] = div(df["ws"], df["g"])
    df["ws40"] = div(df["ws"], df["mp"]) * 40

    if "ts_pct_x" in df.columns and "ts_pct_y" in df.columns:
        df["ts_diff"] = df["ts_pct_x"] - df["ts_pct_y"]
    else:
        df["ts_diff"] = np.nan

    if "efg_pct_x" in df.columns and "efg_pct_y" in df.columns:
        df["efg_diff"] = df["efg_pct_x"] - df["efg_pct_y"]
    else:
        df["efg_diff"] = np.nan

    strength_cols = [c for c in ["win_loss_pct", "net_rtg", "srs"] if c in df.columns]
    if strength_cols:
        temp = df[strength_cols].apply(pd.to_numeric, errors="coerce")
        temp = temp.fillna(temp.median())
        temp_std = temp.std(ddof=0).replace(0, 1)
        df["team_power"] = ((temp - temp.mean()) / temp_std).mean(axis=1)
    else:
        df["team_power"] = np.nan

    df["ws_rookie"] = df["ws"] * df["rookie_flag"]
    df["ws_vet"] = df["ws"] * df["vet_flag"]
    df["ws_hardship"] = df["ws"] * df["hardship_flag"]
    df["pts_rookie"] = df["pts_per_g"] * df["rookie_flag"]
    df["pts_vet"] = df["pts_per_g"] * df["vet_flag"]
    df["pts_hardship"] = df["pts_per_g"] * df["hardship_flag"]

    return df


## Train-Fit Transform Helpers


In [32]:
# Fit PCA on training seasons only, so the 2025 holdout does not affect pca1
def fit_pca(train_df):
    pca_cols = [c for c in PCA_COLS if c in train_df.columns]
    pca_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=1, random_state=42)),
    ])
    pca_pipe.fit(train_df[pca_cols])
    return pca_pipe, pca_cols

# Apply the already-fitted PCA pipeline to train or test rows
def use_pca(df, pca_params):
    pca_pipe, pca_cols = pca_params
    return pca_pipe.transform(df[pca_cols]).ravel()

# Fit numeric scaling and categorical encoding on training data only
def fit_prep(X_train, num_cols, cat_cols):
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    prep = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )
    prep.fit(X_train)
    return prep

# Transform features with the train-fitted preprocessor and return column names
def prep_x(X, prep):
    X_prep = prep.transform(X)
    names = prep.get_feature_names_out().tolist()
    return pd.DataFrame(X_prep, columns=names), names


## Build and Save Processed Artifacts


In [33]:
def main():
    # Locate the repo root whether this notebook is run from /notebooks or the repo root
    repo = Path.cwd()
    if repo.name == "notebooks":
        repo = repo.parent
    raw_dir = repo / "data" / "raw"
    output_dir = repo / "data" / "processed"
    output_dir.mkdir(parents=True, exist_ok=True)

    # Build one merged player-season dataframe for 2021-2025, then add engineered features
    final_df = load_merge(raw_dir)
    final_df = add_features(final_df)
    # Drop missing and zero salary rows before splitting/evaluation
    final_df = final_df.dropna(subset=["salary"])
    final_df = final_df[final_df["salary"] > 0]

    # Elena split: train on historical seasons and reserve 2025 as the holdout set
    train_df = final_df[final_df["year"] <= TRAIN_END_YEAR].copy()
    test_df = final_df[final_df["year"] == TEST_YEAR].copy()

    # Fit PCA on train only, then apply the same PCA transform to both splits
    pca_params = fit_pca(train_df)
    train_df["pca1"] = use_pca(train_df, pca_params)
    test_df["pca1"] = use_pca(test_df, pca_params)

    # Fail early if any required modeling or lookup columns are missing
    missing = [col for col in FEATURES + ["salary", "player", "team", "year"] if col not in train_df.columns]
    if missing:
        raise KeyError(f"Missing required columns after feature engineering: {missing}")

    # Save lookup tables separately so model X files do not include player identifiers
    lookup_cols = ["player", "team", "year", "salary", "group"]
    lookup_train = train_df[lookup_cols].reset_index(drop=True)
    lookup_test = test_df[lookup_cols].reset_index(drop=True)

    # Separate model features from the salary target
    X_train = train_df[FEATURES].reset_index(drop=True)
    y_train = train_df["salary"].reset_index(drop=True)
    X_test = test_df[FEATURES].reset_index(drop=True)
    y_test = test_df["salary"].reset_index(drop=True)

    # Treat group as categorical; all other selected features are numeric
    cat_cols = ["group"]
    num_cols = [c for c in FEATURES if c not in cat_cols]

    # Fit preprocessing on train only and transform 2025 without refitting
    prep = fit_prep(X_train, num_cols, cat_cols)
    X_train_prep, feature_names = prep_x(X_train, prep)
    X_test_prep, test_names = prep_x(X_test, prep)

    # Guard against mismatched train/test columns after one-hot encoding
    if feature_names != test_names:
        raise ValueError("Train and test processed feature names do not match.")

    # Write processed train/test artifacts for metric evaluation
    X_train_prep.to_csv(output_dir / "X_train_processed.csv", index=False)
    pd.DataFrame({"salary": y_train}).to_csv(output_dir / "y_train.csv", index=False)
    X_test_prep.to_csv(output_dir / "X_test_2025_processed.csv", index=False)
    pd.DataFrame({"salary": y_test}).to_csv(output_dir / "y_test_2025.csv", index=False)
    lookup_train.to_csv(output_dir / "player_lookup_train.csv", index=False)
    lookup_test.to_csv(output_dir / "player_lookup_test_2025.csv", index=False)
    pd.Series(feature_names).to_csv(output_dir / "feature_names.csv", index=False, header=["feature"])

    # Print a compact sanity check after saving
    print("Saved train/test processed artifacts")
    print(f"train rows: {len(X_train_prep)}, test rows: {len(X_test_prep)}")
    print(f"processed feature count: {len(feature_names)}")
    print(f"features: {feature_names}")

main()


Saved train/test processed artifacts
train rows: 742, test rows: 223
processed feature count: 25
features: ['avail_rate', 'blk', 'fg', 'fg_per_g', 'fga', 'ft', 'fta', 'g', 'mp', 'pca1', 'pts', 'pts_rookie', 'pts_vet', 'pts_hardship', 'start_rate', 'team_min', 'tov', 'ws_rookie', 'ws_vet', 'ws_hardship', 'group_controlled', 'group_hardship', 'group_rookie', 'group_unknown', 'group_veteran']


## Split Summary

This table documents which seasons are included in each processed split.


In [34]:
repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent

output_dir = repo / "data" / "processed"
X_train_saved = pd.read_csv(output_dir / "X_train_processed.csv")
X_test_saved = pd.read_csv(output_dir / "X_test_2025_processed.csv")
y_train_saved = pd.read_csv(output_dir / "y_train.csv")
y_test_saved = pd.read_csv(output_dir / "y_test_2025.csv")
lookup_train_saved = pd.read_csv(output_dir / "player_lookup_train.csv")
lookup_test_saved = pd.read_csv(output_dir / "player_lookup_test_2025.csv")

lookup_all = pd.concat([
    lookup_train_saved.assign(split="train"),
    lookup_test_saved.assign(split="test_2025"),
])

# Year-level row counts. X and y rows follow the same row order as the lookup files
split_summary = (
    lookup_all
    .groupby(["split", "year"])
    .size()
    .reset_index(name="lookup_rows")
)
split_summary["X_rows_for_year"] = split_summary["lookup_rows"]
split_summary["y_rows_for_year"] = split_summary["lookup_rows"]
split_summary = split_summary[["split", "year", "X_rows_for_year", "y_rows_for_year", "lookup_rows"]]

# File-level total row checks
file_summary = pd.DataFrame([
    {"file": "X_train_processed.csv", "rows": len(X_train_saved)},
    {"file": "y_train.csv", "rows": len(y_train_saved)},
    {"file": "player_lookup_train.csv", "rows": len(lookup_train_saved)},
    {"file": "X_test_2025_processed.csv", "rows": len(X_test_saved)},
    {"file": "y_test_2025.csv", "rows": len(y_test_saved)},
    {"file": "player_lookup_test_2025.csv", "rows": len(lookup_test_saved)},
])

display(split_summary)
display(file_summary)


,split,year,X_rows_for_year,y_rows_for_year,lookup_rows
0,test_2025,2025,223,223,223
1,train,2021,187,187,187
2,train,2022,200,200,200
3,train,2023,176,176,176
4,train,2024,179,179,179


,file,rows
0,X_train_processed.csv,742
1,y_train.csv,742
2,player_lookup_train.csv,742
3,X_test_2025_processed.csv,223
4,y_test_2025.csv,223
5,player_lookup_test_2025.csv,223


Expected split definition:

- Train: 2021-2024 historical seasons
- Test: 2025 holdout season
- The preprocessor and PCA are fit on train only, then applied to 2025.
